# 电商用户行为分析项目**项目背景**  基于淘宝电商平台用户行为数据，分析用户在点击→收藏→加购→购买各环节的行为特征与转化规律，为优化用户运营策略提供数据支撑。**数据说明**  - 数据来源：UserBehavior.csv（淘宝用户行为日志，约1亿条，本分析取前300万条）- 时间范围：2017-07-03 至 2017-12-06- 覆盖用户：29,178人 | 覆盖商品：805,067个- 字段：用户id、商品id、商品类目id、用户行为（pv/buy/cart/fav）、时间戳**分析框架**  1. **用户行为总览** — 整体行为分布与用户活跃度特征2. **转化漏斗分析** — 点击→加购→购买各环节转化率与流失点，下钻至品类维度定位优化方向3. **用户分层分析（RFM）** — 基于R（最近购买间隔）、F（购买次数）、M（行为总量）三维度将用户划分为6个层级，并对比各层的购买力与活跃度特征4. **时间维度分析** — 用户日活跃、周活跃、时段活跃规律，并结合购买转化率定位高效运营时段5. **数据导出** — 为Tableau可视化提供清洗后的结构化数据**工具**：Python（pandas、numpy）| 可视化：Tableau

In [1]:
#导入数据库
import numpy as np
import pandas as pd

# 导入csv文件查看表格结构

In [2]:
#由于原始csv文件没有表头，因此需要手动添加表头
DATA_PATH = 'data/UserBehavior.csv'
columns = ['用户id' , '商品id', '商品类目id', '用户行为', '发生时间']
df = pd.read_csv(DATA_PATH, nrows=3000000, header= None, names = columns)

#查看表结构
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 5 columns):
 #   Column  Dtype 
---  ------  ----- 
 0   用户id    int64 
 1   商品id    int64 
 2   商品类目id  int64 
 3   用户行为    object
 4   发生时间    int64 
dtypes: int64(4), object(1)
memory usage: 114.4+ MB


# 数据清洗

In [3]:
#数据概览——前五行
print('\n列表前五行数据概览:')
print(df.head())

#缺失值查询
print('\n缺失值分布:')
print(df.isnull().sum())
print()

#重复值查询
print('\n重复值数量:')
print(df.duplicated().sum())
df = df.drop_duplicates()

#数据类型转换，将时间列的格式转换为datetime格式，注意时间戳！
df['发生时间'] = pd.to_datetime(df['发生时间'], unit='s')
df = df[df['发生时间'].between('2017-01-01','2017-12-31')]

#替换、修正文本
df['用户行为'] = df['用户行为'].replace({'pv': '点击', 'buy': '购买', 'cart': '加入购物车', 'fav': '收藏商品'})

#最终清洗结果统计
print('\n=====清洗后数据统计=====:')
print(f'清洗后的有效数据量:{df.shape[0]}')
print(f'数据覆盖时间范围:{df['发生时间'].min()}至{df['发生时间'].max()}')
print(f'数据覆盖用户数量:{df['用户id'].nunique()}')
print(f'数据覆盖商品数量：{df['商品id'].nunique()}')
print(f'数据覆盖商品种类{df['商品类目id'].nunique()}')
df


列表前五行数据概览:
   用户id     商品id   商品类目id 用户行为        发生时间
0     1  2268318  2520377   pv  1511544070
1     1  2333346  2520771   pv  1511561733
2     1  2576651   149192   pv  1511572885
3     1  3830808  4181361   pv  1511593493
4     1  4365585  2520377   pv  1511596146

缺失值分布:
用户id      0
商品id      0
商品类目id    0
用户行为      0
发生时间      0
dtype: int64


重复值数量:
1

=====清洗后数据统计=====:
清洗后的有效数据量:2999981
数据覆盖时间范围:2017-07-03 09:24:32至2017-12-06 12:06:46
数据覆盖用户数量:29178
数据覆盖商品数量：805067
数据覆盖商品种类6868


,用户id,商品id,商品类目id,用户行为,发生时间
0,1,2268318,2520377,点击,2017-11-24 17:21:10
1,1,2333346,2520771,点击,2017-11-24 22:15:33
2,1,2576651,149192,点击,2017-11-25 01:21:25
3,1,3830808,4181361,点击,2017-11-25 07:04:53
4,1,4365585,2520377,点击,2017-11-25 07:49:06
...,...,...,...,...,...
2999995,218275,2772813,1577687,点击,2017-11-29 15:10:10
2999996,218275,4654492,1577687,点击,2017-11-29 15:11:08
2999997,218275,2772813,1577687,点击,2017-11-29 15:11:17
2999998,218275,1207527,1577687,点击,2017-11-29 15:12:50


# 用户行为总览

In [4]:
custom_behavior_number = df.groupby('用户行为').size() #用户的行为构成
avg_behavior_number = (df['用户行为'].value_counts() / df['用户id'].nunique()).round() #人均行为数
behavior_ration = df['用户行为'].value_counts(normalize= True) #用户整体行为占比

#数据展示
print(f'各行为数量:{custom_behavior_number.sort_values(ascending= False)}')
print(f'\n用户人均行为数量:{avg_behavior_number.sort_values(ascending= False)}')
print(f'\n整体各行为占比:{behavior_ration.sort_values(ascending=False)}')
print()

#用户活跃度分析
user_activity = df.groupby('用户id').size() #每一位用户的行为数
user_activity_med = user_activity.median()
print(f'\n用户总行为次数中位数{user_activity_med}\n')

percentile = [1,10,20,30,40,50,60,70,80,90,99]
for p in percentile:
    value = user_activity.quantile(p/100)
    print(f'第{p}%分位数是:{value}') 

#寻找80%用户的活跃次数区间
p10 = user_activity.quantile(0.1)
p90 = user_activity.quantile(0.9)
print('\n====80%用户的行为分布====')
print(f'中间80%用户的行为量集中在分布在{p10}--{p90}次之间')

各行为数量:用户行为
点击       2685859
加入购物车     167572
收藏商品       86576
购买         59974
dtype: int64

用户人均行为数量:用户行为
点击       92.0
加入购物车     6.0
收藏商品      3.0
购买        2.0
Name: count, dtype: float64

整体各行为占比:用户行为
点击       0.895292
加入购物车    0.055858
收藏商品     0.028859
购买       0.019991
Name: proportion, dtype: float64


用户总行为次数中位数77.0

第1%分位数是:8.0
第10%分位数是:22.0
第20%分位数是:34.0
第30%分位数是:47.0
第40%分位数是:61.0
第50%分位数是:77.0
第60%分位数是:97.0
第70%分位数是:122.0
第80%分位数是:158.0
第90%分位数是:221.0
第99%分位数是:410.0

====80%用户的行为分布====
中间80%用户的行为量集中在分布在22.0--221.0次之间


![](tableau_image\整体各行为类型占比.png)

# 转化漏斗分析  
## 1.转化率分析

In [5]:
#计算每个行为背后的用户数量
funnel_data = {}
for behavior in ['点击', '收藏商品', '加入购物车', '购买']:
    users = df[df['用户行为'] == behavior]['用户id'].nunique()
    funnel_data[behavior] = users
    print(f'{behavior}:{users}人')
print("""\n设定的原有转化路径是:点击-->收藏商品-->加入购物车-->购买
经过统计发现,收藏行为的人数过少,不符合漏斗分析层层递减的要求
这说明对许多用户而言,收藏不是必经之路,因此将收藏视为一种可选行为
更改后的转化路径为:点击-->加入购物车-->购买""")
print()

#定义新的漏斗路径
new_funnel_data = {}
for behavior in ['点击',  '加入购物车', '购买']:
    users = df[df['用户行为'] == behavior]['用户id'].nunique()
    new_funnel_data[behavior] = users
    print(f'{behavior}:{users}人')
print()
#计算各环节的环比转化率与整体转化率
print('-'*70)
print(f'{'行为环节':<12} {'用户数':10} {'环比转化率':<12} {'整体转化率':<12} ')
print('-'*70)

pre_users = None
clic_users = funnel_data['点击']
for behavior,users in new_funnel_data.items():
    if pre_users is None:
        #点击环节，没有环比转化率和流失率
        cvr = '-'
        overall = '100%'
        loss = '-'
    else:
        #环比转化率 = (本层用户数量/上一层用户数量)
        cvr_value = users/pre_users
        cvr = f'{cvr_value*100:.1f}%'
        #整体转化率 = (本层用户数量/起始层用户数量)
        overall_value = users/clic_users
        overall = f'{overall_value*100:.1f}%'
    print(f'{behavior:<12} {users:<15} {cvr:<15} {overall:<15} ')
    pre_users = users

点击:29049人
收藏商品:11701人
加入购物车:22061人
购买:19902人

设定的原有转化路径是:点击-->收藏商品-->加入购物车-->购买
经过统计发现,收藏行为的人数过少,不符合漏斗分析层层递减的要求
这说明对许多用户而言,收藏不是必经之路,因此将收藏视为一种可选行为
更改后的转化路径为:点击-->加入购物车-->购买

点击:29049人
加入购物车:22061人
购买:19902人

----------------------------------------------------------------------
行为环节         用户数        环比转化率        整体转化率        
----------------------------------------------------------------------
点击           29049           -               100%            
加入购物车        22061           75.9%           75.9%           
购买           19902           90.2%           68.5%           


![](tableau_image\漏斗图.png)

## 2.流失率分析

In [6]:
print('-'*70)
print('流失分析')
print('-'*70)

stage = ['点击', '加入购物车', '购买']
for i in range(1, len(stage)):
    pre_stage = stage[i-1]
    cur_stage = stage[i]
    pre_count = new_funnel_data[pre_stage]
    cur_count = new_funnel_data[cur_stage]
    print(f'{pre_stage}-->{cur_stage}:流失{pre_count - cur_count}人, 流失率:{((pre_count - cur_count)/pre_count)*100:.1f}%')
print()

print("""流失率最严重的环节:点击-->加入购物车
流失率:24.1%
该环节流失了6988人
点击量高但是加入购物车的行为较少
说明在商品宣传环节上没有充分吸引用户,存在进一步优化的空间""")

----------------------------------------------------------------------
流失分析
----------------------------------------------------------------------
点击-->加入购物车:流失6988人, 流失率:24.1%
加入购物车-->购买:流失2159人, 流失率:9.8%

流失率最严重的环节:点击-->加入购物车
流失率:24.1%
该环节流失了6988人
点击量高但是加入购物车的行为较少
说明在商品宣传环节上没有充分吸引用户,存在进一步优化的空间


## 3.按商品种类拆解流失率

In [7]:
#先按照商品类目分组聚合，得到每一种商品的点击、加购、购买的人数
click_users = df[df['用户行为'] == '点击'].groupby('商品类目id')['用户id'].nunique()
cart_users  = df[df['用户行为'] == '加入购物车'].groupby('商品类目id')['用户id'].nunique()
buy_users   = df[df['用户行为'] == '购买'].groupby('商品类目id')['用户id'].nunique()
#将结果整合到一张表中
category_funnel = pd.DataFrame({
    '点击用户数': click_users,
    '加购用户数': cart_users,
    '购买用户数': buy_users
}).fillna(0).astype(int)
#计算每一个中商品的点击-->加购转化率
category_funnel['点击-->加购转化率'] = (category_funnel['加购用户数'] / category_funnel['点击用户数'] * 100).round(1)

#计算出流量较大（点击量大于均值）,但是转化率低于均值的商品，这些往往是优化的重点
category_funnel_hightraffic = category_funnel[category_funnel['点击用户数'] >= category_funnel['点击用户数'].mean()].copy()

#将需要重点优化的商品种类标注出来
category_funnel_hightraffic['是否需要重点优化'] = np.where(
    category_funnel_hightraffic['点击-->加购转化率'] < category_funnel_hightraffic['点击-->加购转化率'].mean(), '需要', '不需要')

category_funnel_hightraffic.sort_values('点击-->加购转化率')

,点击用户数,加购用户数,购买用户数,点击-->加购转化率,是否需要重点优化
商品类目id,,,,,
1686309,123,0,5,0.0,需要
539513,111,0,2,0.0,需要
1676308,222,1,2,0.5,需要
3126258,149,1,1,0.7,需要
903158,144,1,0,0.7,需要
...,...,...,...,...,...
2507053,120,36,13,30.0,不需要
5008221,169,53,28,31.4,不需要
3740459,206,65,39,31.6,不需要


以上品类的商品在 点击-->加购 环节符合**高流量、低转化**的特征，存在很大的增长潜力，需要重点优化，提高整体ROI。  
**运营建议**如下：

- 优化主图/详情页
- 增加促销力度
- 增加推广力度，补充优质买家点评

**效果预估**：

若流失率从24.1%降至20%（保守提升4个百分点），以当前点击用户29,049人为基准，预计新增加购用户约1178人

按当前加购→购买转化率90.2%计算，预计新增购买用户约1061人，购买用户数提升比例约为5.3%


# 用户分层分析(RFM)
## 1.对用户进行分层

In [8]:
#在这里定义R:距离最近一次购买的天数； 
#         F:每个用户购买行为的次数； 
#         M:各行为数量的总和(点击+收藏+加购+购买的总次数)
rfm_analyse = df.copy()
rfm_analyse['发生日期'] = rfm_analyse['发生时间'].dt.normalize()
all_user = rfm_analyse['用户id'].unique()


#----------计算R、F、M----------
#1.计算R
purchas = rfm_analyse[rfm_analyse['用户行为'] == '购买']
reference_date = purchas['发生日期'].max() + pd.Timedelta(days=1)
R = (reference_date - purchas.groupby('用户id')['发生日期'].max()).dt.days
R_all_user_table = R.reindex(all_user,fill_value=999).reset_index().rename(columns = {'发生日期':'最近购买间隔_天'}) #在这里给从未有过购买行为的用户赋值999，代表其在短期内不会进行购买
print('====用户距离最近一次购买的天数====')
print(R_all_user_table)
print()

#2.计算F
F = rfm_analyse[rfm_analyse['用户行为'] == '购买'].groupby('用户id')['用户行为'].size()
F_all_user_table = F.reindex(all_user, fill_value=0).reset_index().rename(columns={'用户行为':'购买次数'}) #在这里给从未有过购买行为的用户赋值0，代表其购买次数为0
print('====用户购买次数====')
print(F_all_user_table)
print()

#3.计算M — 统计每个用户所有行为(点击+收藏+加购+购买)的总次数
M_table = rfm_analyse.groupby('用户id').size().reset_index().rename(columns={0: '行为次数总和'})
print('====用户行为次数总和====')
print(M_table)
print()

#4.合并R、F、M三张表
rfm =R_all_user_table.merge(F_all_user_table, on='用户id', how='inner').merge(M_table, on='用户id', how='inner')
print('====每个用户的R、F、M值====')
print(rfm)
print()

#----------将R、F、M划分为高、中、低三档----------
#R(间隔越短越优):  低间隔→高, 中间隔→中, 高间隔→低
#F(次数越多越优):  低次数→低, 中次数→中, 高次数→高
#M(次数越多越优):  低次数→低, 中次数→中, 高次数→高
rfm['R_tier'] = pd.qcut(rfm['最近购买间隔_天'], q=3, labels=['高','中','低'], duplicates='drop')
rfm['F_tier'] = pd.qcut(rfm['购买次数'], q=3, labels=['低','中','高'], duplicates='drop')
rfm['M_tier'] = pd.qcut(rfm['行为次数总和'], q=3, labels=['低','中','高'], duplicates='drop')

print('====各维度分层人数分布====')
print(rfm[['R_tier', 'F_tier', 'M_tier']].apply(pd.Series.value_counts))
print()

#----------将27种组合映射为6个业务层级----------
rfm['组合'] = rfm['R_tier'].astype(str) + '-' + rfm['F_tier'].astype(str) + '-' + rfm['M_tier'].astype(str)

#27种R-F-M组合 → 6个业务层级的映射规则
#
# 1) 超级活跃忠诚用户 — R高(近期有购买)、F高(购买频次高)，平台的顶级高价值用户
# 2) 高活跃潜力用户   — M高(行为活跃)、但F低(尚未转化)，有较大转化空间的潜力用户
# 3) 新客/轻度用户    — R高(最近来过)、但F低~中(购买不多)，处于观望期的新客或轻度用户
# 4) 活跃但无购买用户 — 近期活跃但F低(极少/无购买)，只看不买型
# 5) 普通用户         — 各维度均处于中等水平，构成平台的基本盘
# 6) 流失用户         — R低(很久没来)、F低(购买少)或F高~中(历史购买力强但沉寂)，已流失或濒临流失
segment_map = {
    # 超级活跃忠诚用户 — R高(近期↔)、F高(购买多)、M不拘
    '高-高-高': '超级活跃忠诚用户',
    '高-高-低': '超级活跃忠诚用户',
    '高-中-高': '超级活跃忠诚用户',
    '中-高-高': '超级活跃忠诚用户',

    # 高活跃潜力用户 — M高(活跃)、R中~高(近期来过)、但F低(还没买)
    '高-低-高': '高活跃潜力用户',
    '中-低-高': '高活跃潜力用户',

    # 新客/轻度用户 — R高(近期来)、F低~中(购买少)、M低~中(行为少)
    '高-低-低': '新客/轻度用户',
    '高-低-中': '新客/轻度用户',
    '高-中-低': '新客/轻度用户',
    '高-中-中': '新客/轻度用户',

    # 活跃但无购买用户 — R中~高(近期来过)、F低(极少购买)
    '中-低-中': '活跃但购买力较低用户',
    '中-低-低': '活跃但购买力较低用户',

    # 普通用户 — 各维度中等，构成平台基本盘
    '高-高-中': '普通用户',
    '中-高-低': '普通用户',
    '中-高-中': '普通用户',
    '中-中-高': '普通用户',
    '中-中-中': '普通用户',
    '中-中-低': '普通用户',

    # 流失用户 — R低(很久没来)，包含F高~中(历史购买力强但已沉寂)和F低(购买少)
    '低-高-高': '流失用户',
    '低-高-中': '流失用户',
    '低-高-低': '流失用户',
    '低-中-高': '流失用户',
    '低-中-中': '流失用户',
    '低-中-低': '流失用户',
    '低-低-高': '流失用户',
    '低-低-中': '流失用户',
    '低-低-低': '流失用户',
}

rfm['用户分层'] = rfm['组合'].map(segment_map).fillna('其他')

#储存每个格子(27种组合)的用户数量 — 转为DataFrame格式
segment_counts = rfm.groupby(['R_tier', 'F_tier', 'M_tier'], observed=False).size().reset_index(name='用户数')
print('====27格用户数量分布====')
print(segment_counts)

#层级分布汇总表
segment_summary = rfm['用户分层'].value_counts().reset_index()
segment_summary.columns = ['用户层级', '用户数']
segment_summary['占比'] = (segment_summary['用户数'] / segment_summary['用户数'].sum() * 100).round(1)
segment_summary = segment_summary.sort_values('用户数', ascending=False).reset_index(drop=True)
print('====层级分布汇总表====')
print(segment_summary.to_string(index=False))

====用户距离最近一次购买的天数====
          用户id  最近购买间隔_天
0            1       999
1          100         6
2         1000       999
3      1000001         1
4      1000004       999
...        ...       ...
29173   218248         6
29174   218250       999
29175   218267         3
29176    21827       999
29177   218275         5

[29178 rows x 2 columns]

====用户购买次数====
          用户id  购买次数
0            1     0
1          100     8
2         1000     0
3      1000001     1
4      1000004     0
...        ...   ...
29173   218248     2
29174   218250     0
29175   218267     1
29176    21827     0
29177   218275     2

[29178 rows x 2 columns]

====用户行为次数总和====
          用户id  行为次数总和
0            1      55
1           13      11
2           19      75
3           21     336
4          100      98
...        ...     ...
29173  1017990      59
29174  1017994      35
29175  1017997      95
29176  1018000     185
29177  1018011      42

[29178 rows x 2 columns]

====每个用户的R、F、M值====
          用户id  最

In [9]:
print('====层级分布汇总表====')
segment_summary

====层级分布汇总表====


,用户层级,用户数,占比
0,流失用户,9339,32.0
1,超级活跃忠诚用户,6013,20.6
2,普通用户,5342,18.3
3,新客/轻度用户,3722,12.8
4,活跃但购买力较低用户,2921,10.0
5,高活跃潜力用户,1841,6.3


![](tableau_image\用户分层.png)

## 2.用户画像描述

In [10]:
#各层用户行为特征画像
user_profile = rfm.merge(
    df.groupby('用户id').agg(
        点击次数=('用户行为', lambda x: (x == '点击').sum()),
        加购次数=('用户行为', lambda x: (x == '加入购物车').sum()),
        活跃天数=('发生时间', lambda x: x.dt.date.nunique())
    ),
    on='用户id', how='left'
)

#按分层汇总各层用户的行为特征
layer_profile = user_profile.groupby('用户分层').agg(
    人均购买次数=('购买次数', 'mean'),
    人均浏览次数=('点击次数', 'mean'),
    人均活跃天数=('活跃天数', 'mean'),
    人均行为次数=('行为次数总和', 'mean'),
    用户数=('用户id', 'count')
).round(1)

In [11]:
print('====各层用户行为画像====')
layer_profile.sort_values('用户数')

====各层用户行为画像====


,人均购买次数,人均浏览次数,人均活跃天数,人均行为次数,用户数
用户分层,,,,,
高活跃潜力用户,1.0,175.2,8.3,190.8,1841
活跃但购买力较低用户,1.0,48.0,6.8,53.9,2921
新客/轻度用户,1.5,45.8,6.3,52.2,3722
普通用户,3.6,78.8,7.6,90.6,5342
超级活跃忠诚用户,5.0,155.5,8.1,175.2,6013
流失用户,0.0,74.6,6.7,81.3,9339


**关键发现**

**1. 用户结构呈两极分化**
- 流失用户占32.0%（9,339人），超级活跃忠诚用户占20.6%（6,013人），两端合计过半
- 中间主力层（普通用户18.3% + 新客12.8% + 活跃但无购买10.0%）合计约41%
- 高活跃潜力用户仅6.3%（1,841人），说明高行为量的未转化用户占比较小

**2. 各层级购买力差距显著**
- 超级活跃忠诚用户人均购买5.0次，是普通用户（3.6次）的1.4倍，新客（1.5次）的3.3倍
- 流失用户购买次数趋近于0，与其定义为"很久未回访"的特征一致
- 高活跃潜力用户虽然购买仅1.0次，但人均浏览次数（175次）和人均行为次数（191次）接近超级活跃用户，是最值得挖掘的潜力群体

**3. 活跃度≠购买力**
- 流失用户人均访问74.6次，活跃天数6.7天，甚至高于新客（45.8次/6.3天）和活跃但购买力较低用户（48.0次/6.8天）
- 说明相当一部分用户虽然频繁访问浏览，但始终没有或很少转化购买，单纯看活跃量会高估其价值

**分层运营策略**

| 用户层级 | 用户画像 | 运营策略 |
|---|---|---|
| **超级活跃忠诚用户** | 高购买频次、高活跃、人均购买5次 | 会员专属权益、新品优先体验、推荐有礼 |
| **高活跃潜力用户** | 行为活跃但购买少、人均浏览175次 | 定向发券刺激首单、限时折扣跟进 |
| **新客/轻度用户** | 近期来过但购买仅1~2次、处于观望期 | 新人专享券、热销榜单推送、降低决策门槛 |
| **普通用户** | 各维度中等，人均购买3.6次，平台基本盘 | 满减凑单推荐、定期促活、会员积分 |
| **活跃但无购买用户** | 只看不买、人均浏览仅48次，缺乏购买动力 | 精准推荐匹配商品、大力度的新人转化券 |
| **流失用户** | 很久未回访、购买次数趋近0，占比32% | 召回短信/推送、大额回归券、爆款推荐 |

![](tableau_image\用户行为画像.png)

# 用户活跃时间维度分析

## 1.各时段用户活跃度分析

In [12]:
#用户11月份活跃度分析
nov = df.copy()
nov['发生日期'] = nov['发生时间'].dt.normalize()
nov = nov[nov['发生日期'].between('2017-11-01', '2017-11-30')]
#日活跃度分析
nov['发生日期'] = nov['发生时间'].dt.normalize()
daily_active_user = nov.groupby('发生日期')['用户id'].nunique()
print('====用户11月份活跃度分析====')
print(f'日均用户活跃{daily_active_user.mean().round()}')
print(f'最高单日活跃{daily_active_user.max()}')
print(f'最低单日活跃{daily_active_user.min()}')
print(f'最高单日活跃日期{daily_active_user.idxmax().date()}')
print(f'最低单日活跃日期{daily_active_user.idxmin().date()}')
print()

#周活跃度分析
nov['星期几'] = nov['发生时间'].dt.dayofweek
weekday_user = nov.groupby('星期几')['用户id'].nunique()
weekday_user.index = ['周一', '周二', '周三', '周四', '周五', '周六', '周日']
print('====周活跃度分布====')
print(weekday_user.sort_values(ascending=False))
print('''观察发现:除了周五外每日的活跃用户数量相差很小,因此建议在除了周五以外的时间进行推送活动。
此外依然要考虑:出现如此大的偏差会不会是数据出了问题。
本项目所采用的数据总计1亿条,为了项目分析上的方便只选取了前300万条数据进行分析,因此样本有可能会出现时序偏差''')

#小时范围活跃度分析
nov['小时'] = nov['发生时间'].dt.hour
hour_user = nov.groupby('小时')['用户id'].nunique()
print('\n====各时段活跃用户数====')
print(hour_user.sort_values(ascending= False))
print(f"""观察数据发现：
用户活跃早高峰:5:00--9:00,活跃人数:{hour_user[5:10].sum()}人
用户活跃午高峰:10:00--15:00,活跃人数:{hour_user[10:16].sum()}人
用户活跃晚高峰:22:00--4:00,活跃人数:{hour_user[hour_user.index >= 22].sum() + hour_user[hour_user.index < 4].sum()}人
建议在以上时段推送业务""")

====用户11月份活跃度分析====
日均用户活跃5540.0
最高单日活跃21766
最低单日活跃1
最高单日活跃日期2017-11-30
最低单日活跃日期2017-11-03

====周活跃度分布====
周四    21810
周三    21417
周日    21179
周二    21017
周一    20982
周六    20889
周五     5533
Name: 用户id, dtype: int64
观察发现:除了周五外每日的活跃用户数量相差很小,因此建议在除了周五以外的时间进行推送活动。
此外依然要考虑:出现如此大的偏差会不会是数据出了问题。
本项目所采用的数据总计1亿条,为了项目分析上的方便只选取了前300万条数据进行分析,因此样本有可能会出现时序偏差

====各时段活跃用户数====
小时
13    15749
12    15180
14    14863
11    14052
7     13632
5     13604
4     13319
8     13319
6     13205
3     13091
2     12993
9     12850
10    12781
1     11504
15    11390
0      9703
23     7939
16     7377
22     4308
17     3741
18     2196
21     1932
19     1535
20     1412
Name: 用户id, dtype: int64
观察数据发现：
用户活跃早高峰:5:00--9:00,活跃人数:66610人
用户活跃午高峰:10:00--15:00,活跃人数:84015人
用户活跃晚高峰:22:00--4:00,活跃人数:59538人
建议在以上时段推送业务


## 2.各时段购买转化率分析

In [13]:
#各时段购买转化率分析（在上方活跃量基础上，增加转化率维度）
nov['小时'] = nov['发生时间'].dt.hour

#计算各小时的活跃用户数和购买用户数
hour_active = nov.groupby('小时')['用户id'].nunique()
hour_buy = nov[nov['用户行为'] == '购买'].groupby('小时')['用户id'].nunique()

hour_conversion = pd.DataFrame({
    '活跃用户数': hour_active,
    '购买用户数': hour_buy
}).fillna(0).astype(int)
#保留数值列用于计算，另加显示列
hour_conversion['购买转化率'] = (hour_conversion['购买用户数'] / hour_conversion['活跃用户数'] * 100).round(1)
hour_conversion['转化率(显示)'] = hour_conversion['购买转化率'].astype(str) + '%'
hour_conversion.index.name = '小时'
print('====各小时购买转化率====')
print(hour_conversion[['活跃用户数', '购买用户数', '转化率(显示)']].to_string())
print()

#按时段聚合，比较不同时间段的转化效果
def period_label(h):
    if 5 <= h <= 9:
        return '早间(5-9点)'
    elif 10 <= h <= 15:
        return '午间(10-15点)'
    elif 16 <= h <= 21:
        return '晚间(16-21点)'
    else:
        return '深夜(22-4点)'

nov['时段'] = nov['小时'].map(period_label)
period_active = nov.groupby('时段')['用户id'].nunique()
period_buy = nov[nov['用户行为'] == '购买'].groupby('时段')['用户id'].nunique()

period_conversion = pd.DataFrame({
    '活跃用户数': period_active,
    '购买用户数': period_buy
}).fillna(0).astype(int)
period_conversion['购买转化率'] = (period_conversion['购买用户数'] / period_conversion['活跃用户数'] * 100).round(1)
period_conversion['转化率(显示)'] = period_conversion['购买转化率'].astype(str) + '%'
print('====各时段购买转化率对比====')
print(period_conversion.loc[['早间(5-9点)', '午间(10-15点)', '晚间(16-21点)', '深夜(22-4点)']][['活跃用户数', '购买用户数', '转化率(显示)']].to_string())

====各小时购买转化率====
    活跃用户数  购买用户数 转化率(显示)
小时                      
0    9703    942    9.7%
1   11504   1385   12.0%
2   12993   1774   13.7%
3   13091   1783   13.6%
4   13319   1779   13.4%
5   13604   1805   13.3%
6   13205   1700   12.9%
7   13632   1773   13.0%
8   13319   1666   12.5%
9   12850   1468   11.4%
10  12781   1364   10.7%
11  14052   1629   11.6%
12  15180   1963   12.9%
13  15749   2059   13.1%
14  14863   1871   12.6%
15  11390   1262   11.1%
16   7377    894   12.1%
17   3741    341    9.1%
18   2196    188    8.6%
19   1535     98    6.4%
20   1412    102    7.2%
21   1932    119    6.2%
22   4308    321    7.5%
23   7939    668    8.4%

====各时段购买转化率对比====
            活跃用户数  购买用户数 转化率(显示)
时段                              
早间(5-9点)    24895   6771   27.2%
午间(10-15点)  26724   7914   29.6%
晚间(16-21点)  10970   1567   14.3%
深夜(22-4点)   25182   6814   27.1%


**用户活跃时间与购买转化分析**

**1. 用户活跃时段分布**

用户活跃度呈现三个明显的高峰时段：
- **早间活跃高峰（5:00-9:00）**— 活跃用户约2.5万人，用户从清晨开始陆续访问
- **午间活跃高峰（10:00-15:00）**— 活跃用户约2.7万人，是一天中最活跃的时段
- **深夜活跃高峰（22:00-4:00）**— 活跃用户约2.5万人，夜间活跃度超预期，与早间高峰相当

而**晚间（16:00-21:00）**活跃用户仅约1.1万人，明显低于其他时段，用户活动主要集中在下午4-5点，随后逐渐减少。

**2. 各时段购买转化率对比**

将活跃度与购买转化率结合分析：
- **午间（10:00-15:00）转化率最高**，达29.6%，是全天购买意愿最强的时段，午休时间用户更倾向于完成购买决策
- **早间（5:00-9:00）和深夜（22:00-4:00）转化率相当**，分别为27.2%和27.1%，均处于高位
- **深夜时段值得关注**：活跃用户数并非最高，但购买转化率与活跃高峰时段持平甚至更高，如1:00-4:00时段,说明深夜来访的用户购买意图更明确，决策路径更短
- **晚间（16:00-21:00）转化率最低**，仅14.3%，约为其他时段的一半，形成低活跃+低购买转化率的特征

**3. 运营建议**

- **午间加大推广力度**：在活跃度和转化率均为最高的时段，增加核心商品曝光和促销推送，最大化转化效率
- **深夜做精准转化**：针对深夜访问用户推出限时优惠，利用其较强的购买意图提高转化
- **晚间优化承接**：16:00-21:00转化偏低，可优化商品推荐策略，或在该时段投放福利预告，引导用户后续下单

![](tableau_image\各时段活跃用户数与转化率.png)

# 导出制表需要的数据

In [14]:
#定义导出文件夹字段名
output_profile = 'tableau_data'

#漏斗数据--变量来源: new_funnel_data（点击/加购/购买的用户数）
funnel_df = pd.DataFrame(list(new_funnel_data.items()),columns=['行为环节', '用户数'])
##添加环比、整体转化率
funnel_df['环比转化率'] = ['-', '75.9%', '90.2%']
funnel_df['整体转化率'] = ['100%', '75.9%', '68.5%']
funnel_df.to_csv(f'{output_profile}/01_漏斗数据.csv', index=False, encoding='utf-8-sig')


#行为类型占比数据--变量来源: custom_behavior_number（点击/收藏/加购/购买的次数）
behavior_df = custom_behavior_number.reset_index()
behavior_df.columns = ['用户行为', '人数']
behavior_df.to_csv(f'{output_profile}/02_行为类型占比数据.csv', index=False, encoding='utf-8-sig')

#用户分层数据--变量来源: rfm['用户分层'].value_counts()
RFM_df = rfm['用户分层'].value_counts().reset_index()
RFM_df.columns = ['用户层级', '人数']
RFM_df.to_csv(f'{output_profile}/03_RFM用户分层.csv', index=False, encoding='utf-8-sig')

#各时段活跃度数据--变量来源: hour_user（0~23点各小时的活跃用户数）
hour_df = hour_user.reset_index()
hour_df.columns = ['小时', '用户活跃数']
hour_df.to_csv(f'{output_profile}/04_各时段活跃用户.csv', index=False, encoding='utf-8-sig')

#各层用户行为画像--变量来源：layer_profile
layer_profile = layer_profile.reset_index()
layer_profile.to_csv(f'{output_profile}/05_各层用户行为画像.csv', index=False,encoding='utf-8-sig')

#各小时用户购买转化率--变量来源
hour_conversion['购买转化率'] = hour_conversion['购买转化率'].round(1).astype('str') + '%'
hour_conversion_df = hour_conversion.reset_index()
hour_conversion_df.to_csv(f'{output_profile}/06_各小时用户购买转化率.csv', index=False, encoding='utf-8-sig')